# 11.8 - Embedding Generation

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

Combine chunking (11.7) and embeddings (11.3): turn chunks into vectors in batches and store them in a vector database. This is the bridge between text and vector search - index build speed, cost, and quality all live here.

## 2. Why Does This Matter?

Embedding quality and efficiency drive retrieval. Batching makes it fast, and storing metadata alongside vectors lets you retrieve with context later.

## 3. Prerequisites

Units 11.3, 11.5, 11.7.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Chunk a corpus, embed it in batches, and store it in Chroma and FAISS
- Verify the index counts match the number of chunks
- Use the embedding helper from earlier notebooks (fallback safe)

## 5. Mental Model

Embedding generation is a factory assembly line: chunks go in one end, vectors come out the other, and the vector DB stores them for fast lookup.

```text
Chunks -> Batch -> Embedding Model -> Vectors + Metadata -> Vector DB Index
```


## 6. Setup + Corpus + Chunking
We reuse the deterministic embedding helper (MiniLM with hash fallback) and build a small FAQ corpus.

In [1]:
# Deterministic embedding helper.
# Loads all-MiniLM-L6-v2 if available; otherwise falls back to a hash-based
# vector so every cell still completes offline. The fallback still gives
# "similar text -> similar vector" behaviour via character-bigram overlap,
# so the demos remain meaningful without the model download.
import hashlib, numpy as np

_DIM = 384


def _hash_embed(texts):
    vecs = np.zeros((len(texts), _DIM))
    for i, t in enumerate(texts):
        bigrams = [t[j:j+2].lower() for j in range(len(t)-1)]
        for bg in bigrams:
            h = int(hashlib.md5(bg.encode()).hexdigest(), 16) % _DIM
            vecs[i, h] += 1.0
        norm = np.linalg.norm(vecs[i]) or 1.0
        vecs[i] = vecs[i] / norm
    return vecs


_model = None
_model_name = "all-MiniLM-L6-v2"


def get_embedder(force_fallback=False):
    """Return a function texts -> np.ndarray (N, dim)."""
    global _model
    if force_fallback:
        return _hash_embed
    if _model is None:
        try:
            from sentence_transformers import SentenceTransformer
            _model = SentenceTransformer(_model_name)
        except Exception as e:
            print("MiniLM unavailable, using hash fallback:", type(e).__name__)
            _model = None
    if _model is None:
        return _hash_embed
    return lambda texts: np.asarray(_model.encode(list(texts), convert_to_numpy=True))


def embed(texts, force_fallback=False):
    fn = get_embedder(force_fallback=force_fallback)
    return np.asarray(fn(texts), dtype=np.float32)


print("embedding dim:", _DIM)
print("backend:", _model_name if get_embedder() != _hash_embed else "hash-fallback")


embedding dim: 384


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4485.89it/s]

backend: all-MiniLM-L6-v2


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS = [
    ("Return policy allows returns within 30 days of purchase. Items must be in original "
     "packaging. Contact support to start a return."),
    ("Shipping takes 5 to 7 business days for standard delivery. Express arrives in 2 to 3 days "
     "for an extra fee."),
    ("Refunds are processed within 5 to 7 business days after we receive the returned item."),
    ("Warranty covers manufacturing defects for one year from the purchase date."),
]
SOURCES = ["policy.pdf", "shipping.md", "policy.pdf", "warranty.txt"]

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40,
                                          separators=["\n\n", "\n", ". ", " ", ""])
chunk_texts, chunk_ids, metas = [], [], []
for d_i, (doc_text, src) in enumerate(zip(CORPUS, SOURCES)):
    for j, c in enumerate(splitter.split_text(doc_text)):
        chunk_texts.append(c)
        chunk_ids.append(f"c{d_i}_{j}")
        metas.append({"source": src, "doc": d_i})
print("chunks:", len(chunk_texts))


chunks: 4


## 7. Batch Embed + Store in Chroma
Embed all chunks in one batch (the helper already batches via numpy) and add them to a Chroma collection with their metadata and ids.

In [3]:
import chromadb

emb_mat = embed(chunk_texts)
print("embedding batch shape:", emb_mat.shape)

client = chromadb.Client()
col = client.create_collection("faq_index", embedding_function=None)
col.add(documents=chunk_texts, embeddings=emb_mat.tolist(), ids=chunk_ids, metadatas=metas)
print("chunks indexed in Chroma:", col.count())


embedding batch shape: (4, 384)


chunks indexed in Chroma: 4


## 8. Store in FAISS (same vectors)
The same vectors go into an in-memory FAISS index. FAISS keeps the raw matrix; we keep a parallel list so we can map an index position back to its text.

In [4]:
import faiss

faiss_index = faiss.IndexFlatIP(emb_mat.shape[1])
faiss_index.add(emb_mat)
print("faiss ntotal:", faiss_index.ntotal)

# parallel lookup: position order = chunk_texts order
lookup = list(zip(chunk_ids, chunk_texts, metas))
q = embed(["how long for a refund"])[0:1]
D, I = faiss_index.search(q, 3)
print("top-3 faiss positions:", I[0])
for pos in I[0]:
    cid, txt, meta = lookup[pos]
    print(f"  [{cid}] {meta['source']}: {txt[:45]}")


faiss ntotal: 4


top-3 faiss positions: [2 0 1]
  [c2_0] policy.pdf: Refunds are processed within 5 to 7 business 
  [c0_0] policy.pdf: Return policy allows returns within 30 days o
  [c1_0] shipping.md: Shipping takes 5 to 7 business days for stand


## 9. Counts Must Match
Sanity check: every chunk became a vector in both stores. Count mismatches mean lost data upstream.

In [5]:
print("chunks            :", len(chunk_texts))
print("chroma count      :", col.count())
print("faiss ntotal      :", faiss_index.ntotal)
assert len(chunk_texts) == col.count() == faiss_index.ntotal, "count mismatch!"
print("all counts match ✓")


chunks            : 4
chroma count      : 4
faiss ntotal      : 4
all counts match ✓



## Common Mistakes

- Embedding one chunk at a time instead of batching (slow).
- Not tracking which model version was used.
- Exceeding token limits on long chunks.
- Not persisting the index (here: ephemeral/in-memory by design).

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Embedding very slow | No batching / model too large | Batch encode, smaller model |
| Token-count errors | Chunks exceed model limit | Truncate or split long chunks |
| Index count < chunks | Some chunks failed to embed | Log and re-embed failures |
| Count mismatch | Metadata/id alignment bug | Verify lists have equal length |

## Best Practices

- Batch embeddings (100-500 chunks at a time).
- Record embedding model name + version with the index.
- Estimate cost before bulk embedding.
- Test on a subset before the whole corpus.
- Persist the index after building (in production).

## Hands-On Practice

1. **Basic:** Embed 5 chunks and store in Chroma.
2. **Guided:** Embed 100 chunks, measure time, check count.
3. **Independent:** Build an incremental pipeline that adds new chunks.
4. **Realistic:** Compare embedding time across 3 models.
5. **Challenge:** Estimate embedding cost for a 10k-doc corpus.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
